# Day 15 & 16 — Model Evaluation and Error Analysis

## Setup — load evaluation sample and ground truth

In [1]:
import os
import pandas as pd

BASE_DIR = os.getcwd()
DATA_DIR = os.path.normpath(os.path.join(BASE_DIR, "..", "data"))

# Build the evaluation sample once (safe to re-run; overwrites the same file)
df = pd.read_csv(os.path.join(DATA_DIR, "clean_jobs.csv"))
sample = df.sample(n=40, random_state=42).reset_index(drop=True)
sample.to_csv(os.path.join(DATA_DIR, "eval_sample.csv"), index=False)

eval_df = pd.read_csv(os.path.join(DATA_DIR, "eval_sample.csv"))
truth_df = pd.read_csv(os.path.join(DATA_DIR, "eval_ground_truth.csv"))
truth_df["true_skills"] = truth_df["true_skills"].apply(lambda x: set(x.split(";")))

print(f"Loaded {len(eval_df)} sample jobs")
print(f"Loaded {len(truth_df)} ground truth rows")
eval_df.head()


Loaded 40 sample jobs
Loaded 40 ground truth rows


,job_id,job_title,company,location,job_description,experience,education,salary,job_type,clean_description
0,JOB06253,DevOps Engineer,Infosys,Bengaluru,We are looking for a DevOps Engineer to join o...,0-1 years,MBA,6-10 LPA,Contract,we are looking for a devops engineer to join o...
1,JOB04685,Full Stack Developer,Amazon,Thane,We are looking for a Full Stack Developer to j...,3-5 years,BCA,4-6 LPA,Internship,we are looking for a full stack developer to j...
2,JOB01732,DevOps Engineer,Capgemini,Ahmedabad,Capgemini is hiring a DevOps Engineer. The rol...,2-3 years,Bachelor's Degree,6-10 LPA,Full-time,capgemini is hiring a devops engineer the role...
3,JOB04743,Web Developer,Fractal Analytics,Gurugram,Fractal Analytics is hiring a Web Developer. T...,5+ years,Bachelor's Degree,4-6 LPA,Full-time,fractal analytics is hiring a web developer th...
4,JOB04522,SQL Developer,IBM,Chennai,IBM is hiring a SQL Developer. The role involv...,0-1 years,MBA,3-5 LPA,Part-time,ibm is hiring a sql developer the role involve...


In [3]:
# Load the skill taxonomy once, used by TF-IDF, NER, and error analysis below
tax = pd.read_csv(os.path.join(DATA_DIR, "skill_taxonomy.csv"))

known_skills_lower = set()
for _, row in tax.iterrows():
    known_skills_lower.add(row["skill"].strip().lower())
    if isinstance(row["aliases"], str) and row["aliases"].strip():
        for alias in row["aliases"].split(";"):
            known_skills_lower.add(alias.strip().lower())

print(f"Loaded {len(tax)} taxonomy rows -> {len(known_skills_lower)} known skill terms (canonical + aliases)")


Loaded 48 taxonomy rows -> 111 known skill terms (canonical + aliases)


## Method 1 — Dictionary / Rule-Based (Day 7)

In [4]:
import sys

# Add the skill_extractor folder to Python's search path so we can import from it
sys.path.append(os.path.join(BASE_DIR, "..", "skill_extractor"))

from rule_based_extractor import extract_skills, load_skill_list

skill_lookup = load_skill_list()  # loads skill_taxonomy.csv into the lookup dict

eval_df["dict_skills"] = eval_df["clean_description"].apply(lambda t: set(extract_skills(t, skill_lookup)))
print(eval_df["dict_skills"].iloc[0])


{'Kubernetes', 'Docker', 'Jenkins', 'AWS', 'Linux'}


## Method 2 — Regex / Phrase Matching (Day 8)

In [5]:
import re

regex_patterns = {
    "Python": r"\bpython(?:\s*3)?\b",
    "SQL": r"\bsql\b",
    "HTML": r"\bhtml\b",
    "CSS": r"\bcss\b",
    "JavaScript": r"\bjavascript\b",
    "Machine Learning": r"\bmachine learning\b",
    "Deep Learning": r"\bdeep learning\b",
    "Natural Language Processing": r"\bnatural language processing\b",
    "Power BI": r"\bpower\s*bi\b",
    "Data Science": r"\bdata science\b",
    # add more skills here as needed, matching your skill_taxonomy.csv
}

def extract_regex_skills(text):
    text = str(text).lower()
    found = set()
    for skill, pattern in regex_patterns.items():
        if re.search(pattern, text):
            found.add(skill)
    return found

eval_df["regex_skills"] = eval_df["clean_description"].apply(lambda t: set(extract_regex_skills(t)))
print(eval_df["regex_skills"].iloc[0])


set()


## Method 3 — NER / Custom EntityRuler (Day 12)

Patterns are matched on `LOWER` (case-insensitive), since `clean_description` was lowercased during Day 4 cleaning. Only entities the ruler itself tagged as `SKILL` are kept, filtering out unrelated entities the base spaCy model finds (e.g. dates, numbers).

In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")

if "entity_ruler" not in nlp.pipe_names:
    ruler = nlp.add_pipe("entity_ruler", before="ner")
    patterns = []
    for _, row in tax.iterrows():
        patterns.append({"label": "SKILL", "pattern": [{"LOWER": row["skill"].lower()}]})
        if isinstance(row["aliases"], str) and row["aliases"].strip():
            for alias in row["aliases"].split(";"):
                patterns.append({"label": "SKILL", "pattern": [{"LOWER": alias.strip().lower()}]})
    ruler.add_patterns(patterns)

print("nlp pipeline:", nlp.pipe_names)

def extract_ner_skills(text):
    doc = nlp(str(text))
    return set(ent.text.lower() for ent in doc.ents if ent.label_ == "SKILL")

eval_df["ner_skills"] = eval_df["clean_description"].apply(extract_ner_skills)
print(eval_df["ner_skills"].iloc[0])


nlp pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'entity_ruler', 'ner']
{'linux', 'docker', 'aws', 'jenkins', 'kubernetes'}


## Method 4 — Transformer NER Baseline (Day 14)

In [7]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Transformer NER pipeline loaded")


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1786.95it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Transformer NER pipeline loaded


In [8]:
def extract_transformer_skills(text):
    results = ner_pipeline(str(text))
    return set(r["word"].lower() for r in results)

eval_df["transformer_skills"] = eval_df["clean_description"].apply(extract_transformer_skills)
print(eval_df["transformer_skills"].iloc[0])


set()


## Method 5 — TF-IDF (Day 9/10)

Top-scoring TF-IDF terms per job (1–3 word phrases) are matched against the taxonomy by checking whether a known skill appears as a **whole word inside** each top phrase, since n-gram phrases like `"skills include aws"` won't exact-match the single word `"aws"`.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 3), stop_words="english")
tfidf_matrix = vectorizer.fit_transform(eval_df["clean_description"])
feature_names = vectorizer.get_feature_names_out()

def tfidf_skills_for_row(row_idx, top_n=15):
    row_vec = tfidf_matrix[row_idx].toarray().flatten()
    top_indices = row_vec.argsort()[-top_n:]
    top_terms = [feature_names[i] for i in top_indices if row_vec[i] > 0]

    matched_skills = set()
    for term in top_terms:
        for skill in known_skills_lower:
            if skill in term.split():  # skill matches a whole word inside the phrase
                matched_skills.add(skill)
    return matched_skills

eval_df["tfidf_skills"] = [tfidf_skills_for_row(i) for i in range(len(eval_df))]
print(eval_df["tfidf_skills"].iloc[0])


{'linux', 'docker', 'aws', 'jenkins'}


**Note:** this word-containment check only catches single-word skills (e.g. `"aws"`, `"python"`). Multi-word taxonomy skills like `"machine learning"` won't be caught this way, since they're never a single token in `term.split()` — a known limitation of TF-IDF's bag-of-words output, worth naming explicitly in the Day 15 report.

## Day 15 — Precision, Recall, F1 per Method

In [10]:
def compute_metrics(predicted_sets, true_sets):
    tp = fp = fn = 0
    for pred, true in zip(predicted_sets, true_sets):
        pred = {p.strip().lower() for p in pred}
        true = {t.strip().lower() for t in true}
        tp += len(pred & true)
        fp += len(pred - true)
        fn += len(true - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return round(precision, 3), round(recall, 3), round(f1, 3)

methods = {
    "Dictionary": eval_df["dict_skills"],
    "Regex": eval_df["regex_skills"],
    "TF-IDF": eval_df["tfidf_skills"],
    "NER": eval_df["ner_skills"],
    "Transformer": eval_df["transformer_skills"],
    # add "ML" once you decide how to adapt the Day 13 classifier to per-job skill sets
}

results = []
for name, preds in methods.items():
    p, r, f1 = compute_metrics(preds, truth_df["true_skills"])
    results.append({"Method": name, "Precision": p, "Recall": r, "F1": f1})

comparison_df = pd.DataFrame(results)
print(comparison_df)
comparison_df.to_csv(os.path.join(DATA_DIR, "model_comparison.csv"), index=False)


        Method  Precision  Recall     F1
0   Dictionary      0.938   0.985  0.961
1        Regex      1.000   0.385  0.556
2       TF-IDF      0.926   0.560  0.698
3          NER      0.943   0.750  0.836
4  Transformer      0.000   0.000  0.000


### Confusion Matrix (optional, per-skill)

In [11]:
from sklearn.metrics import multilabel_confusion_matrix
import numpy as np

all_skills = sorted(set().union(*truth_df["true_skills"]))

def to_binary_matrix(skill_sets, all_skills):
    return np.array([[1 if s in skills else 0 for s in all_skills] for skills in skill_sets])

y_true = to_binary_matrix(truth_df["true_skills"], all_skills)
y_pred = to_binary_matrix(eval_df["ner_skills"], all_skills)  # swap to whichever method to inspect

cm = multilabel_confusion_matrix(y_true, y_pred)
# cm[i] gives [[TN, FP], [FN, TP]] for all_skills[i]


## Day 16 — Error Analysis

In [12]:
def norm(s):
    return s.strip().lower()

error_rows = []
for i, row in eval_df.iterrows():
    text = row["clean_description"]
    true_skills = {norm(s) for s in truth_df.loc[i, "true_skills"]}
    predicted = {norm(s) for s in row["ner_skills"]}  # swap to whichever method you're analyzing

    # False negatives: a true skill was missed entirely
    for skill in true_skills - predicted:
        error_rows.append({"text": text[:80], "actual": skill, "predicted": "(none)"})

    # False positives: a predicted skill isn't actually in the ground truth
    for skill in predicted - true_skills:
        error_rows.append({"text": text[:80], "actual": "(none)", "predicted": skill})

print(f"Collected {len(error_rows)} mismatches")


Collected 59 mismatches


In [13]:
def guess_category(row):
    actual, predicted = row["actual"], row["predicted"]

    if predicted == "(none)":
        return "False Negative"
    if actual == "(none)":
        return "False Positive"

    # one is a substring of the other -> likely a partial match (e.g. "python" vs "python programming")
    if actual in predicted or predicted in actual:
        return "Partial Entity"

    # simple spelling-closeness check (very small edit distance) -> likely a typo, not a real miss
    if abs(len(actual) - len(predicted)) <= 2 and sum(a != b for a, b in zip(actual, predicted)) <= 2:
        return "Spelling Problem"

    return "Wrong Entity"  # default bucket for anything else -> review manually

error_df = pd.DataFrame(error_rows)
error_df["error_type"] = error_df.apply(guess_category, axis=1)

print(error_df["error_type"].value_counts())
error_df.head(10)


error_type
False Negative    50
False Positive     9
Name: count, dtype: int64


,text,actual,predicted,error_type
0,we are looking for a full stack developer to j...,node.js,(none),False Negative
1,we are looking for a full stack developer to j...,(none),js,False Positive
2,we are looking for a full stack developer to j...,(none),node,False Positive
3,we are looking for a data analyst to join our ...,power bi,(none),False Negative
4,we are looking for a marketing analyst to join...,power bi,(none),False Negative
5,we are looking for a marketing analyst to join...,marketing analytics,(none),False Negative
6,we are looking for a data analyst to join our ...,power bi,(none),False Negative
7,we are looking for a database administrator to...,database administration,(none),False Negative
8,we are looking for a database administrator to...,(none),database,False Positive
9,we are looking for a data analyst to join our ...,power bi,(none),False Negative


### Manual review pass — refine ambiguous "Wrong Entity" rows

In [14]:
review_needed = error_df[error_df["error_type"] == "Wrong Entity"]
print(f"{len(review_needed)} rows need manual review")
review_needed


0 rows need manual review


,text,actual,predicted,error_type


In [15]:
# Example manual corrections — adjust based on what review_needed actually shows
error_df.loc[error_df["actual"] == "ml", "error_type"] = "Synonym Problem"
error_df.loc[error_df["predicted"] == "aws cloud", "error_type"] = "Partial Entity"

# Flag skills the taxonomy doesn't contain at all as "Unknown Skill"
error_df.loc[
    (error_df["error_type"] == "Wrong Entity")
    & (~error_df["actual"].isin(known_skills_lower))
    & (error_df["actual"] != "(none)"),
    "error_type"
] = "Unknown Skill"

print(error_df["error_type"].value_counts())


error_type
False Negative    50
False Positive     9
Name: count, dtype: int64


### Save the Day 16 deliverable

In [16]:
final_error_df = error_df.rename(columns={
    "text": "Text",
    "actual": "Actual",
    "predicted": "Predicted",
    "error_type": "Error",
})[["Text", "Actual", "Predicted", "Error"]]

OUTPUT_PATH = os.path.join(DATA_DIR, "Error_Analysis_Report.xlsx")
final_error_df.to_excel(OUTPUT_PATH, index=False, sheet_name="Errors")

print(f"Saved {len(final_error_df)} error rows to {OUTPUT_PATH}")


Saved 59 error rows to c:\Users\Admin\Desktop\Internship Task\Job Skill extraction\Job_Skill_Extraction\data\Error_Analysis_Report.xlsx
